In [ ]:
print("hello world")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


In [ ]:
data = pd.read_csv(
    "/home/ahad-memon/Desktop/MediCure_AI/Dataset/salesweekly.csv"
)
print("First 5 rows of the dataset:")
print(data.head(5))

print(data.shape)

print("Columns in the dataset: ")
print(data.columns.tolist())

print("Data types of each column:")
print(data.dtypes)

print("Data set information:")
print(data.info())

print("\n Missing values in each column:")
print(data.isnull().sum())

data["datum"] = pd.to_datetime(data["datum"], errors="coerce")

data= data.sort_values(by="datum").reset_index(drop=True)

print("\nDate range:")
print(data["datum"].min(), "to", data["datum"].max())

print("\nDuplicate dates:", data["datum"].duplicated().sum())


In [ ]:
# Check whether dates are weekly and continuous
date_gaps = data["datum"].diff().dropna()

print("\nDate gap counts:")
print(date_gaps.value_counts().head())

print("\nUnexpected date gaps:")
print(data.loc[
    data["datum"].diff().dt.days.ne(7) &
    data["datum"].diff().notna(),
    ["datum"]
])

# Check negative sales values
sales_columns = [
    "M01AB", "M01AE", "N02BA", "N02BE",
    "N05B", "N05C", "R03", "R06"
]

print("\nNegative sales values:")
print((data[sales_columns] < 0).sum())

# Check zero-sales counts
print("\nZero-sales observations:")
print((data[sales_columns] == 0).sum())

In [16]:
sales_columns = [
    "M01AB", "M01AE", "N02BA", "N02BE",
    "N05B", "N05C", "R03", "R06"
]

Test_weeks= 52
LAGS= [1,2,3,4,8,13,26,52]
results=[]

for category in sales_columns:
    series=data[category].astype(float)
    features=pd.DataFrame(index=data.index)
    for lag in LAGS:
        features[f"lag_{lag}"]=series.shift(lag)
    features["target"]=series
    model_data=features.dropna().reset_index(drop=True)
    x=model_data.drop(columns=["target"])
    y=model_data["target"]
    split= len(model_data)-Test_weeks
    x_train, x_test= x.iloc[:split], x.iloc[split:]
    y_train, y_test= y.iloc[:split], y.iloc[split:]
    baseline_pred= x_test["lag_1"].to_numpy()

    #ML model 1
    model= RandomForestRegressor(
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=3, 
        random_state=42
    )

    model.fit(x_train, y_train)
    ml_pred= model.predict(x_test)
    baseline_mae= mean_absolute_error(y_test, baseline_pred)
    ml_mae= mean_absolute_error(y_test, ml_pred)
    results.append({
        "category": category,
        "baseline_mae": baseline_mae,
        "random_forest_mae": ml_mae,
        "improvement": (
             (baseline_mae-ml_mae) / baseline_mae * 100
             if baseline_mae>0 else np.nan
        )
    })
reseults_df= pd.DataFrame(results)

print("\n Forecast evaluation (last 52 weeks): ")
print(reseults_df.round(3).to_string(index=False))
print("\n Mean MAE accross categories: ")
print(
    reseults_df[["baseline_mae", "random_forest_mae"]].mean().round(3)
)




 Forecast evaluation (last 52 weeks): 
category  baseline_mae  random_forest_mae  improvement
   M01AB         8.017              6.508       18.814
   M01AE         7.315              7.236        1.077
   N02BA         5.426              4.531       16.501
   N02BE        40.450             36.477        9.821
    N05B        13.219             10.154       23.187
    N05C         3.212              2.378       25.961
     R03        23.620             20.925       11.411
     R06         6.585              5.953        9.598

 Mean MAE accross categories: 
baseline_mae         13.48
random_forest_mae    11.77
dtype: float64
